# LAB 09 — Tokenization và label alignment

Phân tích 40 câu đầu CoNLL-2003 test set. Chiến lược A chỉ gán nhãn cho subword đầu tiên. Những vị trí còn lại nhận `-100`.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from transformers import AutoTokenizer
from ner_utils import ID_TO_LABEL, ROOT, load_conll2003, tokenize_and_align_labels
from reporting import analyze_tokenization

dataset = load_conll2003()
test = dataset["test"]
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-cased", use_fast=True)

## 1. So sánh word và subword

Các special tokens xuất hiện trong `input_ids` nhưng không tương ứng word nào.

In [ ]:
sample = test[0]
encoded = tokenizer(sample["tokens"], is_split_into_words=True, truncation=True, max_length=256)
word_ids = encoded.word_ids()
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
display(pd.DataFrame({"token": tokens, "input_id": encoded["input_ids"], "attention_mask": encoded["attention_mask"], "word_id": word_ids}))

## 2. Alignment nhãn BIO

Strategy A sử dụng nhãn của word cho subword đầu. Những subword tiếp theo và special tokens không đóng góp trực tiếp vào loss.

In [ ]:
aligned = tokenize_and_align_labels({"tokens": [sample["tokens"]], "ner_tags": [sample["ner_tags"]]}, tokenizer, strategy="first")
label_ids = aligned["labels"][0]
display(pd.DataFrame({"token": tokens, "word_id": word_ids, "label_id": label_ids,
                      "label": [ID_TO_LABEL[x] if x != -100 else "IGNORE" for x in label_ids]}))

## 3. Đối chiếu strategy B

Strategy B lan truyền nhãn word sang các subword, đổi B-X của các subword sau thành I-X.

In [ ]:
propagated = tokenize_and_align_labels({"tokens": [sample["tokens"]], "ner_tags": [sample["ner_tags"]]}, tokenizer, strategy="propagate")
display(pd.DataFrame({"token": tokens, "first": aligned["labels"][0], "propagate": propagated["labels"][0]}))

## 4. Experiment 1 — 40 câu test

R là tổng số subword tokens thuộc word chia cho số original words. Special tokens bị loại khỏi phép tính.

In [ ]:
rows = analyze_tokenization(test, tokenizer, minimum=40)
analysis = pd.DataFrame(rows)
display(analysis.head(20))

## 5. So sánh câu dài, câu ngắn, entity và non-entity

In [ ]:
word_splits = pd.read_csv(ROOT / "artifacts" / "word_splits.csv")
display(analysis.groupby("Sentence group", as_index=False).agg(sentences=("Sentence", "count"), mean_R=("R", "mean"), total_words=("Words", "sum"), split_words=("Words split", "sum")))
display(word_splits.groupby("Type", as_index=False).agg(words=("Word", "count"), mean_subwords=("Subwords", "mean"), split_rate=("Split", "mean")))

## 6. Các word thường bị chia subword

In [ ]:
display(word_splits.loc[word_splits["Split"]].groupby("Word").size().sort_values(ascending=False).head(15).rename("frequency").to_frame())

Kết quả được ghi tại `artifacts/tokenization_analysis.csv`, `artifacts/word_splits.csv` và `artifacts/tokenization_analysis.md`.